# 🎬 Video Chef — Character Replacement (Wan 2.2 Animate-14B)

Replace the person in an input video with a character from a single reference image, preserving
motion, facial expressions, and scene lighting (via the Relighting LoRA).

**Model: `Wan2.2-Animate-14B`** (Apache 2.0)

| Colab tier | GPU | Feasibility |
|---|---|---|
| Pay-as-you-go / Pro | **L4 24 GB** | ✅ works with `--offload_model True --convert_model_dtype` (slow) |
| Pay-as-you-go / Pro+ | **A100 40 GB** | ✅ recommended — ~2–3× faster |
| Free | T4 16 GB | ❌ too little VRAM, skip |

> Runtime → Change runtime type → **L4** or **A100**.
> Pipeline: preprocess (pose + face + mask + bg) → generate → save.


In [ ]:
# @title 0. Check GPU
!nvidia-smi

In [ ]:
# @title 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/Wan-Inputs /content/drive/MyDrive/Wan2.2/outputs
print('Drive mounted.')
print('Weights expected at: /content/drive/MyDrive/Wan2.2/Wan2.2-Animate-14B')
print('Put your input video + character image in: /content/drive/MyDrive/Wan-Inputs/')

In [ ]:
# @title 2. Clone Wan 2.2 + install animate deps
%cd /content
![ -d Wan2.2 ] || git clone --depth 1 https://github.com/Wan-Video/Wan2.2.git
%cd /content/Wan2.2
# Install core and specialized requirements
!pip install -q -e .
!pip install -q -r requirements.txt
!pip install -q -r requirements_animate.txt
# Fixes for tokenizers and common huggingface issues
!pip install -q ftfy regex "huggingface_hub[cli]"
print('Setup done. Animation dependencies installed.')

In [ ]:
# @title 3. Download Animate-14B weights (stored on Drive)
REPO_ID = "Wan-AI/Wan2.2-Animate-14B"
CKPT_DIR = "/content/drive/MyDrive/Wan2.2/Wan2.2-Animate-14B"

from huggingface_hub import snapshot_download
import os, shutil

os.makedirs(CKPT_DIR, exist_ok=True)

# Space check
_, _, free = shutil.disk_usage("/")
print(f"Local disk check: {free // (2**30)} GB free.")

if os.path.exists(os.path.join(CKPT_DIR, "config.json")):
    print(f"✅ Model found at {CKPT_DIR}. Skipping download.")
else:
    print(f"Downloading {REPO_ID} → {CKPT_DIR}")
    print("This is ~30 GB; first run takes 20-40 min. Resume enabled.")
    snapshot_download(
        repo_id=REPO_ID,
        local_dir=CKPT_DIR, 
        local_dir_use_symlinks=False, 
        resume_download=True
    )
    print("Done.")

In [ ]:
# @title 4. Select inputs
# @markdown Paths to your input video and reference character image.
# @markdown Default: pulls first matching files from /content/drive/MyDrive/Wan-Inputs/
import os, glob, shutil

VIDEO_PATH = ""      # @param {type:"string"}
IMAGE_PATH = ""      # @param {type:"string"}

INPUTS_DIR = "/content/drive/MyDrive/Wan-Inputs"
if not VIDEO_PATH:
    mp4s = glob.glob(f"{INPUTS_DIR}/*.mp4")
    VIDEO_PATH = mp4s[0] if mp4s else ""
if not IMAGE_PATH:
    imgs = glob.glob(f"{INPUTS_DIR}/*.jpg") + glob.glob(f"{INPUTS_DIR}/*.jpeg") + glob.glob(f"{INPUTS_DIR}/*.png")
    IMAGE_PATH = imgs[0] if imgs else ""

assert VIDEO_PATH and os.path.exists(VIDEO_PATH), f"Video not found: {VIDEO_PATH}"
assert IMAGE_PATH and os.path.exists(IMAGE_PATH), f"Image not found: {IMAGE_PATH}"

WORK_DIR  = "/content/Wan2.2/examples/wan_animate/replace"
PROC_DIR  = f"{WORK_DIR}/process_results"
os.makedirs(WORK_DIR, exist_ok=True)
shutil.copy(VIDEO_PATH, f"{WORK_DIR}/video.mp4")
shutil.copy(IMAGE_PATH, f"{WORK_DIR}/image{os.path.splitext(IMAGE_PATH)[1]}")
print(f"VIDEO: {VIDEO_PATH}\nIMAGE: {IMAGE_PATH}\nWORK:  {WORK_DIR}")

In [ ]:
# @title 5. Preprocess (pose + face + mask + background)
# @markdown Area of the generated video (W H). 1280x720 is the max reliable on L4; use 832x480 for faster tests.
RES_W = 832   # @param {type:"integer"}
RES_H = 480   # @param {type:"integer"}

%cd /content/Wan2.2
img_ext = os.path.splitext(IMAGE_PATH)[1]
import time; t0=time.time()
!python ./wan/modules/animate/preprocess/preprocess_data.py \
    --ckpt_path "{CKPT_DIR}/process_checkpoint" \
    --video_path "{WORK_DIR}/video.mp4" \
    --refer_path "{WORK_DIR}/image{img_ext}" \
    --save_path "{PROC_DIR}" \
    --resolution_area {RES_W} {RES_H} \
    --iterations 3 --k 7 --w_len 1 --h_len 1 \
    --replace_flag
print(f"Preprocess elapsed: {(time.time()-t0)/60:.1f} min")

In [ ]:
# @title 6. Run Wan-Animate (replacement mode)
import time
%cd /content/Wan2.2
t0=time.time()
# Replacement mode + Relighting LoRA + VRAM-friendly offload
!python generate.py --task animate-14B \
    --ckpt_dir "{CKPT_DIR}" \
    --src_root_path "{PROC_DIR}/" \
    --refert_num 1 \
    --replace_flag \
    --use_relighting_lora \
    --offload_model True \
    --convert_model_dtype \
    --use_flash_attn false \
    --t5_cpu
print(f"Inference elapsed: {(time.time()-t0)/60:.1f} min")

In [ ]:
# @title 7. Show result + save to Drive
import glob, shutil, os, time
from IPython.display import HTML
from base64 import b64encode

vids = sorted(glob.glob("/content/Wan2.2/*.mp4"), key=os.path.getmtime, reverse=True)
assert vids, "No mp4 produced — check inference output."
latest = vids[0]

ts = time.strftime("%Y%m%d_%H%M%S")
drive_out = f"/content/drive/MyDrive/Wan2.2/outputs/replace_{ts}.mp4"
shutil.copy(latest, drive_out)
print(f"Saved: {drive_out}")

data_url = "data:video/mp4;base64," + b64encode(open(latest, 'rb').read()).decode()
HTML(f'<video width=720 controls src="{data_url}"></video>')